The IMDB dataset is a collection of movie reviews (text) and a label, `1` means the reviewer likes the movie and `0` means they did not like it. 

# Part I (FastText Classification)

## Data preprocessing

In [3]:
import fasttext
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)
word_index = imdb.get_word_index()
rev_index = {v + 1: k for k, v in word_index.items()}
rev_index.update({0: "<PAD>"})


def decode(x):
    return " ".join(rev_index.get(i, "?") for i in x)

Create two files: `imdb_train.txt` and `imdb_test.txt`.

* `imdb_train.txt` should contain the first 10,000 examples.

* `imdb_test.txt` should contain the first 5,000 examples.

Each line in these files should follow the format:

`f"__label__{y} {decode(x)}"`

Where:

* `y` is the label (e.g., 0 or 1),

* `x` is the input (e.g., a sequence of token IDs or bytes),

`decode(x)` converts `x` into a readable text string.
To learn more about FastText Embedding please see: https://fasttext.cc/docs/en/supervised-tutorial.html. 

In [4]:
# Create the FastText training file
with open("imdb_train.txt", "w", encoding="utf-8") as f:
    for x, y in zip(x_train[:10000], y_train[:10000]):
        f.write(f"__label__{y} {decode(x)}\n")

# Create the FastText test file
with open("imdb_test.txt", "w", encoding="utf-8") as f:
    for x, y in zip(x_test[:5000], y_test[:5000]):
        f.write(f"__label__{y} {decode(x)}\n")

print("Files created successfully.")

Files created successfully.


In [5]:
# Display a few examples from the training file
with open("imdb_train.txt", "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().strip()[:500])

__label__1 ? was not for it's self joke professional disappointment see already pretending their staged a every so found of his movies it's third plot good episodes the in who guess wasn't of doesn't a again plot find the poor let her a again vegas trouble with fight like that oh a big good for to watching essentially but was not a fat centers turn a not well how this for it's self like bad as that natural a not with starts with this for david movie the of only moments this br special br films o
__label__0 ? long younger long closure will far of to shock ears roots scene have a miss br over news was school film that where setting can't think of without being physically on was were however br all you is a day better then a acting in vote helps of relationship a best in who battling a making in 3 bottom a over in a man elm but a inside of most one cheesy if from a 5 in has it's like delivers saving often a struggle have overboard of actually i assistant the a younger in long fantastic is

Now, train a model that learns the word2vec embeddings of the data, averages them for each review, and classifies the review as either 0 or 1. You can use the function `fasttext.train_supervised("imdb_train.txt")` for this purpose.

The train_supervised function accepts several parameters:

* `lr: Learning rate`
* `epoch: Number of training iterations`
* `wordNgrams: Maximum length of word n-gram`
* `dim: Size of word vectors`

First, try training the model using the default parameters. Then, experiment by tweaking these parameters to improve performance. Compare the results and see which configuration yields better accuracy.


In [6]:
# Train FastText using default parameters
fasttext_default = fasttext.train_supervised(input="imdb_train.txt")

# Evaluate on the test set
default_result = fasttext_default.test("imdb_test.txt")

print("Default FastText results:")
print("Number of examples:", default_result[0])
print("Precision:", default_result[1])
print("Recall:", default_result[2])

Default FastText results:
Number of examples: 5000
Precision: 0.8072
Recall: 0.8072


In [7]:
def fasttext_accuracy(model, filename):
    correct = 0
    total = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)

            true_label = parts[0].replace("__label__", "")

            text = parts[1]

            predicted_label = model.predict(text)[0][0]
            predicted_label = predicted_label.replace("__label__", "")

            if predicted_label == true_label:
                correct += 1

            total += 1

    return correct / total

In [8]:
default_accuracy = fasttext_accuracy(fasttext_default, "imdb_test.txt")

print(f"Default FastText accuracy: {default_accuracy:.4f}")

Default FastText accuracy: 0.8072


In [9]:
# Train FastText with modified parameters
fasttext_tuned = fasttext.train_supervised(
    input="imdb_train.txt", lr=0.5, epoch=10, wordNgrams=2, dim=100
)

# Evaluate
tuned_result = fasttext_tuned.test("imdb_test.txt")

print("Tuned FastText results:")
print("Number of examples:", tuned_result[0])
print("Precision:", tuned_result[1])
print("Recall:", tuned_result[2])

Tuned FastText results:
Number of examples: 5000
Precision: 0.8804
Recall: 0.8804


In [10]:
tuned_accuracy = fasttext_accuracy(fasttext_tuned, "imdb_test.txt")

print(f"Tuned FastText accuracy: {tuned_accuracy:.4f}")

Tuned FastText accuracy: 0.8804


# Part II (Classification with LSTM)

## Data preprocessing

In [11]:
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
from keras.preprocessing.sequence import pad_sequences

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=200)

x_train, x_test = pad_sequences(x_train, maxlen=200), pad_sequences(x_test, maxlen=200)

Create an `LSTM model` where the first layer is an `Embedding layer` with a vocabulary size of `200`, and each word is mapped to a `128-dimensional` vector. The second layer should be an LSTM layer with `64 units`, followed by a final Dense layer with a `sigmoid activation function` for binary classification. Compile the model using the `binary_crossentropy` loss function, the `adam` optimizer, and `acc` as the evaluation metric. Train the model for three `epochs` with a batch size of `64`. What happens when you change the dimension of the `Embedding layer` and the number of `epochs`?

In [12]:
# Create the LSTM model
model = Sequential(
    [
        Embedding(input_dim=200, output_dim=128, input_length=200),
        LSTM(64),
        Dense(1, activation="sigmoid"),
    ]
)

# Explicitly build the model
model.build(input_shape=(None, 200))
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["acc"])
model.summary()

c:\Users\Vesal\Documents\Code\Python\Jupyter\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │        25,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,073 (293.25 KB)

 Trainable params: 75,073 (293.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Train the LSTM model for 3 epochs
history = model.fit(x_train, y_train, epochs=3, batch_size=64)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 27s 66ms/step - acc: 0.6383 - loss: 0.6276
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 33s 85ms/step - acc: 0.6957 - loss: 0.5797
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step - acc: 0.7313 - loss: 0.5409


# Part III (Classification with BERT)

## Data preprocessing

In [7]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import torch

dataset = load_dataset(
    "stanfordnlp/imdb", split="train[:100000]+test[:5000]"
).train_test_split(test_size=0.2)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True)


dataset = dataset.map(tokenize, batched=True)
dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 6000/6000 [00:05<00:00, 1081.81 examples/s]


* Upload a bert model called `distilbert-base-uncased` with `AutoModelForSequenceClassification`, make sure to set `num_labels` to `2`. Because our sentiment analysis has two classes. 
* Set the training arguments using `TrainingArguments`, make sure to set `num_train_epochs` and `output_dir` and set `report_to="none"`.
* Create a `Trainer` and set its `model` and `args` to what you defined in th previous steps. Make sure to set `train_dataset=dataset["train"]`, and `eval_dataset=dataset["test"]`. 
* Call the trainer by `.train()`
* Once the model is trained compute its accuracy using the following method (what happens when you increase `num_train_epochs`):

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 892.82it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results", num_train_epochs=1, report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

Step,Training Loss
500,0.396607
1000,0.324536
1500,0.295918
2000,0.270150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
500,0.396607
1000,0.324536
1500,0.295918
2000,0.270150
2500,0.268798
3000,0.242762


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.2997951329549154, metrics={'train_runtime': 1362.6628, 'train_samples_per_second': 17.613, 'train_steps_per_second': 2.202, 'total_flos': 3179217567744000.0, 'train_loss': 0.2997951329549154, 'epoch': 1.0})

In [ ]:
from sklearn.metrics import accuracy_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds)}


trainer.compute_metrics = compute_metrics
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}")

Training Loss,Validation Loss,Step,Accuracy
0.242762,0.252720,3000,0.924333


Accuracy: 0.9243
